# E3 — Reinforcement learning of the view-selection policy (Colab)

**When to run:** after the robot has recorded the view library (`collect_views.py`) and you have run, on the Jetson,
`python3 build_view_dataset.py` and `python3 sync.py pack-for-colab`.

**Upload two files:** the framework zip (your `mycobot_framework` folder, with the latest `views.py`,
`train_views.py`, `analyze_views.py`, `plot_all.py`, `geo_attn.py`, `sync.py`) and the `to_colab_<stamp>.zip`.

**What happens:** a transformer policy and an MLP ablation are trained by REINFORCE on the real recorded views
(every reward comes from an image the robot took), tested on held-out placements against the baselines and the
oracle, and exported for the Jetson (NumPy, parity-checked).

**Outputs to download:** `to_jetson_<stamp>.zip` (policies for `run_view_policy.py`) and `results_views.zip`.

Every cell prints what it did; if something is missing it stops with a message instead of producing nothing.

In [ ]:
# 1) Upload the framework zip and the to_colab_*.zip (or put them in /content beforehand)
import glob, os
have = glob.glob('/content/*.zip')
if not any(os.path.basename(f).startswith('to_colab') for f in have) or len(have) < 2:
    from google.colab import files
    files.upload()
print('zips in /content:', [os.path.basename(f) for f in glob.glob('/content/*.zip')])

In [ ]:
# 2) Unpack, find the framework folder, add the library, print the inventory
import glob, os, zipfile, json, sys, numpy as np
zips = sorted(glob.glob('/content/*.zip'))
data_zips = [z for z in zips if os.path.basename(z).startswith('to_colab')]
fw_zips = [z for z in zips if not os.path.basename(z).startswith(('to_colab', 'to_jetson', 'results_views'))]
assert data_zips, 'no to_colab_*.zip uploaded (run  python3 sync.py pack-for-colab  on the Jetson)'
assert fw_zips, 'no framework zip uploaded (zip your mycobot_framework folder)'
for z in fw_zips:
    zipfile.ZipFile(z).extractall('/content/fw')
cands = sorted(glob.glob('/content/fw/**/train_views.py', recursive=True), key=len)
assert cands, 'train_views.py not found in the framework zip'
ROOT = os.path.dirname(cands[0]); os.chdir(ROOT); sys.path.insert(0, ROOT)
zipfile.ZipFile(data_zips[-1]).extractall(ROOT)
print('framework folder:', ROOT, '| data:', os.path.basename(data_zips[-1]))
for f in ['views.py', 'train_views.py', 'geo_attn.py', 'analyze_views.py', 'plot_all.py', 'sync.py']:
    print('  %-18s %s' % (f, 'ok' if os.path.exists(f) else 'MISSING'))
assert os.path.exists('data/view_library.npz'), 'data/view_library.npz missing: run build_view_dataset.py on the Jetson, then pack-for-colab'
import views
src = open('views.py').read()
z = np.load('data/view_library.npz'); meta = json.loads(str(z['meta']))
vs = views.ViewSet.from_meta(meta) if hasattr(views.ViewSet, 'from_meta') else None
assert vs is not None, 'views.py is an old version: copy the latest views.py into the framework zip'
names, poses, split = z['names'], z['poses'], z['split']
TILTED = bool(vs.rot.any())
print('\nplacements: %d %s | test (held out): %s' % (len(names), {str(k): int(v) for k, v in zip(*np.unique(poses, return_counts=True))},
      [str(n) for n in names[split == 1]]))
print('candidate views: %d = %d cells x %d levels; heights %s mm; tilted levels %s'
      % (vs.K, vs.P, vs.L, [float(x) for x in vs.dz], [tuple(float(x) for x in vs.rpy[l]) for l in range(vs.L) if vs.rot[l]] or 'none'))
print('policy inputs per view (TOK_DIM): %d' % views.TOK_DIM)
n_min = min(np.unique(poses, return_counts=True)[1])
if len(names) < 8 or (split == 1).sum() == 0:
    print('\nWARNING: too few placements for a real result (need ~15+ per pose). Only the SMOKE TEST below is meaningful.')

In [ ]:
# 3) Helper: run a script and stream its output; stop on failure
import subprocess, time
def run(args, tail=None):
    t = time.time(); print('$ python', ' '.join(args), flush=True)
    p = subprocess.Popen([sys.executable] + args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if tail is None or 'iter' not in line or ' 0 ' in line or line.startswith('iter  2000'):
            print(line, end='')
    p.wait()
    if p.returncode != 0:
        print(''.join(lines[-30:])); raise RuntimeError('%s failed (exit %d)' % (args[0], p.returncode))
    print('-> done in %.1f min\n' % ((time.time() - t) / 60))
    return ''.join(lines)
import torch; print('torch', torch.__version__, '| GPU' if torch.cuda.is_available() else '| CPU (fine: the model is small)')

In [ ]:
# 4) SMOKE TEST (2-3 min): the whole training/evaluation/export path on a few iterations
run(['train_views.py', '--arch', 'attn', '--seed', '99', '--iters', '50', '--eval-every', '25'])
print('smoke test OK: parity line above must say < 1e-4')

In [ ]:
# 5) MAIN EXPERIMENT: transformer (attention between candidate views) vs identical MLP, 3 seeds each
ITERS = 2000
for arch in ['attn', 'mlp']:
    for s in [0, 1, 2]:
        run(['train_views.py', '--arch', arch, '--seed', str(s), '--iters', str(ITERS)], tail=True)

In [ ]:
# 6) ABLATIONS of the transformer: no pose input, no time input, and (if recorded) no tilted views
abl = [['--no-pose'], ['--no-time']] + ([['--no-tilt']] if TILTED else [])
for flags in abl:
    for s in [0, 1, 2]:
        run(['train_views.py', '--arch', 'attn', '--seed', str(s), '--iters', str(ITERS)] + flags, tail=True)

In [ ]:
# 7) Results on HELD-OUT placements (mean +- SD over seeds) and choice of the deployed policy
import pandas as pd, shutil
rows = []
for f in sorted(glob.glob('results_views/train_*/test_summary.json')):
    S = json.load(open(f)); tag = os.path.basename(os.path.dirname(f))[6:]
    if '_s99' in tag:
        continue
    var = S['arch'] + ''.join(x for x, on in (('-nopose', S.get('no_pose')), ('-notime', S.get('no_time')),
                                              ('-notilt', S.get('no_tilt'))) if on)
    r = dict(variant=var, seed=S['seed'], policy_pct_of_oracle=S['policy_pct_of_oracle'])
    for c in S['comparisons']:
        r[c['method'] + '_pct'] = c['pct_of_oracle']; r['p_vs_' + c['method']] = c['wilcoxon_p']
    rows.append(r)
assert rows, 'no training results found'
R = pd.DataFrame(rows)
cols = ['policy_pct_of_oracle'] + [c for c in R.columns if c.endswith('_pct') and c != 'oracle_pct']
summary = R.groupby('variant')[cols].agg(['mean', 'std']).round(1)
display(summary)
print('Wilcoxon p (policy vs method), per seed:'); display(R[['variant', 'seed'] + [c for c in R.columns if c.startswith('p_vs_')]].round(4))
# deploy the seed with the best VALIDATION score (never chosen on test results)
for arch in ['attn', 'mlp']:
    cand = {s: f'deploy/view_policy_{arch}_s{s}.npz' for s in [0, 1, 2] if os.path.exists(f'deploy/view_policy_{arch}_s{s}.npz')}
    best = max(cand, key=lambda s: float(np.load(cand[s])['info_val_pct']))
    shutil.copy(cand[best], f'deploy/view_policy_{arch}.npz')
    print('%s: deploy seed %d (validation %.1f%% of oracle)' % (arch, best, float(np.load(cand[best])['info_val_pct'])))

In [ ]:
# 8) Learning curves, statistics and the paper figures
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
for f in sorted(glob.glob('results_views/train_*/curve.csv')):
    tag = os.path.basename(os.path.dirname(f))[6:]
    if '_s99' in tag: continue
    c = pd.read_csv(f); ax.plot(c['iter'], c['val_pct_of_oracle'], label=tag, lw=1)
ax.set_xlabel('training iteration'); ax.set_ylabel('validation, % of oracle'); ax.legend(fontsize=6, ncol=2); ax.grid(alpha=.3)
os.makedirs('paper/figs', exist_ok=True); fig.tight_layout(); fig.savefig('paper/figs/fig_learning_curves.png', dpi=200); plt.show()
for script in ['analyze_views.py', 'plot_all.py']:
    try:
        run([script])
    except Exception as e:
        print('NOTE:', e, '(continuing; send me the output above)')
from IPython.display import Image, display as show
for f in sorted(glob.glob('paper/figs/*.png')):
    print(f); show(Image(f, width=600))

In [ ]:
# 9) Pack the policies for the Jetson and the results for the paper, then download
run(['sync.py', 'pack-for-jetson'])
shutil.make_archive('/content/results_views', 'zip', ROOT, 'results_views')
shutil.make_archive('/content/paper_figs', 'zip', ROOT, 'paper/figs')
from google.colab import files
files.download(sorted(glob.glob(os.path.join(ROOT, 'to_jetson_*.zip')))[-1])
files.download('/content/results_views.zip'); files.download('/content/paper_figs.zip')
print('On the Jetson:  python3 sync.py unpack to_jetson_<stamp>.zip  then run_view_policy.py (E4)')